In [ ]:
# AGI Bench: Proactive/Retroactive Interference (v3)
!pip install -q protobuf==5.29.6 kaggle-benchmarks numpy 2>/dev/null


## Cognitive Science Rationale

**Interference** — when competing learned material disrupts correct application of a target system — is a fundamental memory phenomenon (Underwood, 1957; Postman, 1961). This benchmark measures interference resistance by presenting multiple competing rule systems simultaneously and testing whether the model can selectively apply the correct one.

### Key Design Insight (v3)
Previous versions provided rules independently per prompt, making interference impossible (each prompt was a fresh context with no competing information). v3 creates interference **within the prompt** by presenting multiple competing rule systems together.

### Scoring (v3)
Three difficulty tiers with weighted composite:
- **Easy (0.15):** Simple substitution rules + 1 dissimilar distractor
- **Medium (0.35):** Context-dependent rules + 1 similar distractor
- **Hard (0.50):** Multi-pass chained rules + 2 similar distractors + interleaved items

Per tier: score = 0.30 × control + 0.70 × interference_accuracy

### References
- Underwood (1957): Proactive inhibition in retention
- Postman (1961): Retroactive inhibition
- Anderson (2003): Retrieval-induced forgetting
- Wickens (1972): Release from proactive interference


In [ ]:
import random
import hashlib
from dataclasses import dataclass, field


@dataclass
class RuleSystem:
    """A generated rule system with examples."""
    name: str
    description: str
    rules: list[str]
    examples: list[dict]
    test_items: list[dict]
    difficulty: int
    n_rules: int
    domain: str


def _make_rng(seed: str) -> random.Random:
    h = int(hashlib.sha256(seed.encode()).hexdigest(), 16)
    return random.Random(h)


def generate_symbol_system(seed: str = "sym_default", difficulty: int = 1) -> RuleSystem:
    rng = _make_rng(seed)
    shapes = ["\u25b3", "\u25cb", "\u25a1", "\u25c7", "\u2605", "\u2b21", "\u2b1f", "\u25bd"]

    if difficulty == 1:
        src = rng.sample(shapes[:4], 3)
        dst = rng.sample(shapes[4:], 3) + [rng.choice(shapes[4:])]
        mapping = dict(zip(src, dst[:3]))
        rules = [f"Replace {s} with {d}" for s, d in mapping.items()]
        rules.append("All other symbols stay the same")
        def apply_rules(seq):
            return [mapping.get(s, s) for s in seq]

    elif difficulty == 2:
        src = rng.sample(shapes[:5], 4)
        dst = rng.sample(shapes[4:], 3) + [rng.choice(shapes)]
        mapping = dict(zip(src[:3], dst[:3]))
        pair_rule = (src[0], src[1], dst[3])
        rules = [f"Replace {s} with {d}" for s, d in mapping.items()]
        rules.append(f"EXCEPTION: {pair_rule[0]} followed by {pair_rule[1]} \u2192 both become {pair_rule[2]}")
        rules.append("All other symbols stay the same")
        def apply_rules(seq):
            result = []
            i = 0
            while i < len(seq):
                if i + 1 < len(seq) and seq[i] == pair_rule[0] and seq[i + 1] == pair_rule[1]:
                    result.extend([pair_rule[2], pair_rule[2]])
                    i += 2
                else:
                    result.append(mapping.get(seq[i], seq[i]))
                    i += 1
            return result

    else:
        src = rng.sample(shapes[:6], 5)
        dst = rng.sample(shapes, 5)
        mapping1 = {src[0]: dst[0], src[1]: dst[1]}
        mapping2 = {dst[0]: dst[2]}
        cond = src[2]
        extra_map = {src[3]: dst[3]}
        rules = [f"Pass 1: Replace {s} with {d}" for s, d in mapping1.items()]
        rules.append(f"Pass 2: Replace {list(mapping2.keys())[0]} with {list(mapping2.values())[0]}")
        rules.append(f"IF the sequence contains {cond}: also replace {src[3]} with {dst[3]}")
        rules.append("All other symbols stay the same throughout")
        def apply_rules(seq):
            result = [mapping1.get(s, s) for s in seq]
            result = [mapping2.get(s, s) for s in result]
            if cond in seq:
                result = [extra_map.get(s, s) for s in result]
            return result

    all_items = []
    for _ in range(25):
        length = rng.randint(3, 6)
        seq = [rng.choice(shapes[:5]) for _ in range(length)]
        output = apply_rules(seq)
        all_items.append({"input": " ".join(seq), "output": " ".join(output)})

    seen = set()
    unique_items = []
    for item in all_items:
        if item["input"] not in seen:
            seen.add(item["input"])
            unique_items.append(item)

    rng.shuffle(unique_items)
    n_examples = min(15, len(unique_items) - 5)
    examples = unique_items[:n_examples]
    test_items = unique_items[n_examples:n_examples + 5]

    return RuleSystem(
        name=f"SymbolTransform-{seed}",
        description="Apply symbol transformation rules to input sequences",
        rules=rules, examples=examples, test_items=test_items,
        difficulty=difficulty, n_rules=len(rules), domain="symbol",
    )


In [ ]:
import kaggle_benchmarks as kbench
import re
import json


@dataclass
class InterfAnswer:
    answer: str


def normalize_output(text: str) -> str:
    text = text.strip().lower()
    text = re.sub(r'\s+', ' ', text)
    return text


def check_output(model_output: str, expected: str) -> bool:
    m = normalize_output(model_output)
    e = normalize_output(expected)
    return e in m or m in e


def _format_system(system, max_examples: int = 6) -> str:
    text = f"**{system.name}**\nRules:\n"
    for r in system.rules:
        text += f"  - {r}\n"
    text += "Examples:\n"
    for ex in system.examples[:max_examples]:
        text += f"  {ex['input']} \u2192 {ex['output']}\n"
    return text


def _test_items(llm, system, context: str, prefix: str) -> float:
    correct = 0
    items = system.test_items
    for ti, test_item in enumerate(items):
        with kbench.chats.new(f"{prefix}_{ti}"):
            prompt = (
                context +
                f"\nInput: {test_item['input']}\n\n"
                f"Respond with ONLY: {{\"answer\": \"<output>\"}}"
            )
            try:
                result = llm.prompt(prompt, schema=InterfAnswer)
                answer = result.answer
            except Exception:
                raw = llm.prompt(prompt)
                try:
                    parsed = json.loads(re.search(r'\{.*\}', raw, re.DOTALL).group())
                    answer = str(parsed.get("answer", raw))
                except Exception:
                    answer = raw
            if check_output(answer, test_item["output"]):
                correct += 1
    return correct / len(items) if items else 0


# System definitions (3 tiers)
EASY_TARGET = generate_symbol_system("v3_easy_target", difficulty=1)
EASY_DISTRACT = generate_symbol_system("v3_easy_distract", difficulty=1)
MED_TARGET = generate_symbol_system("v3_med_target", difficulty=2)
MED_DISTRACT = generate_symbol_system("v3_med_distract", difficulty=2)
HARD_TARGET = generate_symbol_system("v3_hard_target", difficulty=3)
HARD_DISTRACT1 = generate_symbol_system("v3_hard_dist1", difficulty=3)
HARD_DISTRACT2 = generate_symbol_system("v3_hard_dist2", difficulty=3)


def run_tier(llm, target, distractors, prefix):
    target_text = _format_system(target)
    ctrl_context = (
        f"You have learned the following rule system:\n\n{target_text}\n"
        f"Apply the **{target.name}** rules to this input."
    )
    control = _test_items(llm, target, ctrl_context, f"{prefix}_ctrl")

    n_ex = 4 if len(distractors) >= 2 else 6
    target_text_short = _format_system(target, max_examples=n_ex)
    all_text = target_text_short
    for d in distractors:
        all_text += "\n" + _format_system(d, max_examples=n_ex)

    interleave = ""
    if len(distractors) >= 2:
        interleave = "\nYou recently processed these items from other systems:\n"
        for d in distractors:
            for di in d.test_items[:2]:
                interleave += f"  [{d.name}] {di['input']} \u2192 {di['output']}\n"

    interf_context = (
        f"You have learned ALL of these rule systems:\n\n{all_text}\n"
        f"{interleave}\n"
        f"Now apply ONLY the **{target.name}** rules "
        f"(ignore all other systems) to this input."
    )
    interference = _test_items(llm, target, interf_context, f"{prefix}_interf")

    tier_score = 0.30 * control + 0.70 * interference
    return {"control": control, "interference": interference, "tier_score": round(tier_score, 4)}


@kbench.task(name="learning_interference")
def learning_interference(llm) -> float:
    """Proactive & Retroactive Interference Benchmark (v3).

    Measures interference resistance: can the model apply rules from a target
    system while competing systems' rules are also present in context?

    Three tiers:
    - Easy (0.15): Simple target + 1 dissimilar distractor
    - Medium (0.35): Moderate target + 1 similar distractor
    - Hard (0.50): Complex target + 2 similar distractors + interleaved items

    Per tier: score = 0.30 * control + 0.70 * interference_accuracy
    Composite = 0.15 * easy + 0.35 * medium + 0.50 * hard
    """
    print("\n" + "=" * 60)
    print("LEARNING INTERFERENCE BENCHMARK v3")
    print("=" * 60)

    print("\n--- EASY TIER ---")
    easy = run_tier(llm, EASY_TARGET, [EASY_DISTRACT], "easy")
    print(f"  Control: {easy['control']:.1%}, Interference: {easy['interference']:.1%}, Score: {easy['tier_score']:.4f}")

    print("\n--- MEDIUM TIER ---")
    medium = run_tier(llm, MED_TARGET, [MED_DISTRACT], "med")
    print(f"  Control: {medium['control']:.1%}, Interference: {medium['interference']:.1%}, Score: {medium['tier_score']:.4f}")

    print("\n--- HARD TIER ---")
    hard = run_tier(llm, HARD_TARGET, [HARD_DISTRACT1, HARD_DISTRACT2], "hard")
    print(f"  Control: {hard['control']:.1%}, Interference: {hard['interference']:.1%}, Score: {hard['tier_score']:.4f}")

    score = round(0.15 * easy["tier_score"] + 0.35 * medium["tier_score"] + 0.50 * hard["tier_score"], 4)
    score = max(0.0, min(1.0, score))

    print(f"\nCOMPOSITE SCORE: {score:.4f}")
    return score


learning_interference.run(llm=kbench.llm)
